# Customized 1-in-X Workflow for Custom Climate Metrics

This notebook computes 1-in-X return values for a custom climate metric across a clipped region. As an example, it calculates a custom Effective Temperature (T<sub>eff</sub>) metric and runs extreme-value analysis on it.

The notebook starts with a GWL-based approach, and then shows a similar analysis using a time-based approach.

Runtime: 20 Minutes

### Steps
1. Define the T<sub>eff</sub> formula as a Dataset-level function and register it with `register_user_function`.
2. Configure the 1-in-X analysis (return periods, distribution, GWL or time window).
3. Define the clip region and other query parameters.
4. Run the full pipeline in a single `ClimateData` chain — clip → unit conversion → GWL subsetting → metric_calc.


#### Step 0: Import libraries

In [ ]:
import pandas as pd
import xarray as xr

from climakitae.new_core.user_interface import ClimateData
from climakitae.new_core.derived_variables import register_user_function
from climakitae.new_core.processors.export import Export


For quiet output, set `VERBOSE = -1` in the cell below. For silent output use `-2`.

In [ ]:
VERBOSE = 0

#### Step 1: Define and Register the Custom Metric

We define a customized version of effective temperature (T<sub>eff</sub>):
```
T_eff = 0.7 × T_max(0) + 0.003 × T_min(0) × T_max(1) + 0.002 × T_min(1) × T_max(2)
```

**Where:** `T_max(0)` / `T_min(0)` are the current day's max/min, `T_max(1)` / `T_min(1)` are 1-day lags, and `T_max(2)` is the 2-day lag of max.

`register_user_function` expects a **Dataset-in / Dataset-out** function. After registration, `"effective_temperature"` becomes a first-class variable that the `ClimateData` pipeline can query by name — the clip, unit conversion, and GWL processors will run before the derived variable is computed, keeping the intermediate arrays small.


In [ ]:
def calc_effective_temperature(ds: xr.Dataset) -> xr.Dataset:
    """Compute effective temperature (Teff) and add it to the Dataset.

    Teff = 0.7*Tmax(0) + 0.003*Tmin(0)*Tmax(1) + 0.002*Tmin(1)*Tmax(2)

    Parameters
    ----------
    ds : xr.Dataset
        Dataset containing ``t2max`` and ``t2min`` DataArrays with a
        ``time`` or ``time_delta`` dimension.

    Returns
    -------
    xr.Dataset
        Input Dataset with ``effective_temperature`` added. The first two
        timesteps are NaN due to the shift operations.
    """
    tasmax = ds["t2max"]
    tasmin = ds["t2min"]

    if "time_delta" in tasmax.dims:
        time_dim = "time_delta"
    elif "time" in tasmax.dims:
        time_dim = "time"
    else:
        raise ValueError("Data must have either 'time' or 'time_delta' dimension")

    teff = (
        0.7 * tasmax
        + 0.003 * tasmin * tasmax.shift({time_dim: 1})
        + 0.002 * tasmin.shift({time_dim: 1}) * tasmax.shift({time_dim: 2})
    )
    ds["effective_temperature"] = teff
    ds["effective_temperature"].attrs = {
        "long_name": "Effective temperature (energy-demand weighted lag)",
        "units": tasmax.attrs.get("units", ""),
        "description": (
            "0.7*Tmax(0) + 0.003*Tmin(0)*Tmax(1) + 0.002*Tmin(1)*Tmax(2); "
            "first two timesteps are NaN due to lagging."
        ),
    }
    return ds


register_user_function(
    name="effective_temperature",
    depends_on=["t2max", "t2min"],
    func=calc_effective_temperature,
    description=(
        "Effective temperature: 0.7·Tmax(0) + 0.003·Tmin(0)·Tmax(1) "
        "+ 0.002·Tmin(1)·Tmax(2)"
    ),
)


#### Step 2: Use the Built-In `metric_calc` Processor for 1-in-X Analysis

Rather than maintaining a hand-rolled batch routine, we use the built-in [`metric_calc`](https://climakitae.readthedocs.io/en/latest/climate-data-interface/processors/metric_calc/) processor in `one_in_x` mode. It handles:

- Block-maxima extraction (annual or sub-annual)
- GEV / Gumbel / Generalized Pareto distribution fitting (vectorized across simulations and spatial points)
- Adaptive memory-aware batching across the simulation dimension
- Kolmogorov–Smirnov goodness-of-fit p-values

The processor returns a Dataset with `return_values` and `p_values` data variables, indexed by simulation and spatial coordinate (one entry per location selected via the `clip` processor).

`one_in_x` config keys used below:

| Key | Value |
|-----|-------|
| `return_periods` | `[10, 100]` — 1-in-10 and 1-in-100 year events |
| `distribution` | `"gev"` — Generalized Extreme Value |
| `extremes_type` | `"max"` — annual maxima |
| `event_duration` | `(1, "day")` — daily block size |


#### Step 3: Calculate 1-in-X Events for Effective Temperature

First, we'll set-up a number of useful arguments, including *location*, *data variable retrieval*, *1-in-X approach return periods*, and *batch size*. 

In [ ]:
# Data selections -- modify for a different custom metric or region
variable = "effective_temperature"
units = "degF"               # target units passed to the convert_units processor
activity_id = "WRF"          # downscaling method: "LOCA2" (statistical) or "WRF" (dynamical)
table_id = "day"             # timescale: "day" (daily), "mon" (monthly), "1hr" (hourly)
grid_label = "d03"           # resolution: "d03" (3km), "d02" (9km), "d01" (45km)
experiment_id = "ssp370"     # GWL approach scenario: "ssp245", "ssp370", or "ssp585"
clip_region = "Sacramento County"  # must match a region in the catalog's "clip" collection

# 1-in-X specifications -- modify for custom return periods
return_periods = [10, 100]

# GWL approach
wls = [2.0, 2.5]

#### Step 4: Build the 1-in-X pipeline

The full analysis runs in a single `ClimateData` chain. The registered `"effective_temperature"` variable tells the pipeline to fetch `t2max` and `t2min`, then processors execute in priority order:

1. `clip` — spatial subsetting (pushes selection down to intake-esm so only the clipped cells are materialized)
2. `convert_units` — unit conversion
3. `warming_level` — GWL-based temporal subsetting
4. *(derived variable computed here on the small clipped arrays)*
5. `metric_calc` — block-maxima extraction → GEV fitting → return values


In [ ]:
# Shared metric_calc config
ONE_IN_X_CONFIG = {
    "return_periods": return_periods,
    "distribution": "gev",
    "extremes_type": "max",
    "event_duration": (1, "day"),
}

#### Step 5: Execute the pipeline

Two variants are provided below — **GWL-based** and **time-based**. Comment out whichever you do not need.

> **Runtime note**: All real computation happens inside `metric_calc`. The processor reports progress via `tqdm` while iterating over its internal sim/spatial batches.


In [ ]:
# GWL-based pipeline
result_gwl = (
    ClimateData(verbosity=VERBOSE)
    .catalog("cadcat")
    .activity_id(activity_id)
    .institution_id("UCLA")
    .table_id(table_id)
    .grid_label(grid_label)
    .variable("effective_temperature")
    .processes({
        "clip": clip_region,
        "convert_units": units,
        "warming_level": {"warming_levels": wls},
        "metric_calc": {"one_in_x": ONE_IN_X_CONFIG},
    })
    .get()
)

result_gwl

#### Step 6: Export the results

Export via the `Export` processor. This can alternatively be added directly to the `.processes({...})` dict in Step 5.

| Key | Value |
|-----|-------|
| `filename` | base output name (no extension) |
| `file_format` | `"NetCDF"`, `"Zarr"`, or `"CSV"` |
| `mode` | `"local"` (default) or `"s3"` (Zarr only) |
| `export_method` | `"data"` (default) or `"skip_existing"` |


In [ ]:
EXPORT_CONFIG = {
    "filename": "teff_gwl",
    "file_format": "NetCDF",  # or "Zarr" / "CSV"
    "export_method": "skip_existing",
}

result_gwl = Export(EXPORT_CONFIG).execute(result_gwl, context={})

In [ ]:
result_gwl.median(dim="sim").return_values.plot.contourf(
    x="lon",
    y="lat",
    col="one_in_x",
    row="warming_level",
    levels=50,
);

## Time-based Approach

The following code runs a similar analysis, but using a time-based approach instead of a GWL based-approach.

In [ ]:
TIME_SLICES = {
    "mid-century": (2030, 2060),
    "late-century": (2070, 2100),
}

In [ ]:
# Time-based pipeline

results_time = {}

for label, time_slice in TIME_SLICES.items():
    results_time[label] = (
        ClimateData(verbosity=VERBOSE)
        .catalog("cadcat")
        .activity_id(activity_id)
        .institution_id("UCLA")
        .experiment_id(experiment_id)  
        .table_id(table_id)
        .grid_label(grid_label)
        .variable("effective_temperature")
        .processes({
            "clip": clip_region,
            "convert_units": units,
            "time_slice": time_slice,
            "metric_calc": {"one_in_x": ONE_IN_X_CONFIG},
        })
        .get()
    )

In [ ]:
#merge results into one Dataset with a new "time_period" dimension
result_time = xr.concat(
    results_time.values(),
    dim=pd.Index(results_time.keys(), name="time_period")
)

In [ ]:
# Plotting the time-based results
result_time.median(dim="sim").return_values.plot.contourf(
    x="lon",
    y="lat",
    col="one_in_x",
    row="time_period",
    levels=50,
);

This notebook demonstrates how to:
1. Define a custom climate metric (T<sub>eff</sub>) as a Dataset-level function and register it with `register_user_function`.
2. Run the full 1-in-X analysis in a single `ClimateData` pipeline — spatial clipping, unit conversion, GWL subsetting, and `metric_calc` are all chained together.
3. Export results and visualize return values as a faceted contour plot.
